
# HackRF One — Frequency Error Calibration (Robust Pipeline)

## Goal

Estimate and correct the crystal oscillator offset of a HackRF One by using
FM broadcast stations as stable frequency references. The offset is modelled as
a single scalar (PPM) that scales linearly with frequency, recovered via
least-squares regression across multiple stations and multiple capture cycles.

---

## Physical model

A constant fractional clock error produces a frequency offset that is strictly
proportional to the carrier frequency:

```
Δf_i = f_measured_i − f_nominal_i = α · f_nominal_i
```

where `α = ppm_error × 1e-6` is the unknown scalar to estimate.
This is a **zero-intercept linear model**: all stations share the same `α`,
so opposite-sign offsets at two stations cannot both be genuine clock error —
at least one is corrupted by an external disturbance.

---

## Pipeline

### Stage 1 — Spectrum acquisition with max-hold stabilisation

- Tune the HackRF One to cover the full FM broadcast band (~88–108 MHz) using
  a 20 MHz sample rate centred at 98 MHz.
- Perform **N = 10** successive PSD captures using Welch's method
  (Hamming window, 50 % overlap, RBW = 10 kHz).
- Apply a **max-hold** across all captures: each frequency bin retains the
  highest power value observed over the N sweeps.
- Rationale: max-hold preserves persistent carriers while suppressing
  transient noise, producing a stable representative spectrum.

---

### Stage 2 — DC spike exclusion

- Mask a symmetric window of **±2.5 %** of the total bandwidth around the
  centre bin before any peak search.
- Apply the mask only to the search copy of the PSD; the unmasked PSD is
  retained for centroid computation.
- Rationale: the HackRF One's inherent DC offset artifact sits at DC and would
  otherwise be misidentified as a strong carrier.

---

### Stage 3 — Noise floor estimation and detection threshold

- Estimate the noise floor as the **10th percentile** of all power values in
  the masked PSD. This is a robust estimator: it is unaffected by the presence
  of strong carriers or isolated spikes.
- Set the detection threshold at **noise_floor + 6 dB** (a fixed margin above
  the floor, not a fraction of dynamic range).
- Only bins exceeding this threshold are forwarded to peak detection.

---

### Stage 4 — Peak detection and bandwidth validation

- Run `scipy.signal.find_peaks` on the masked PSD with a minimum bin
  separation corresponding to **400 kHz** (one FM channel guard band).
- For each candidate peak, measure its bandwidth at 80 % relative height using
  `peak_widths`.
- Retain only candidates whose bandwidth falls within **50 kHz – 300 kHz**,
  the characteristic spectral width of a broadcast FM carrier.
- Discard candidates outside this range (narrowband interferers, wideband
  artifacts, residual DC).
- Keep the **top 5 candidates** ranked by peak power for centroid estimation.

---

### Stage 5 — Sub-bin centroid estimation

For each validated peak, compute a **power-weighted frequency centroid**
within a ±100 kHz window centred on the peak bin:

```
f_centroid = Σ(f_k · P_k) / Σ(P_k)     for k in [peak − 100 kHz, peak + 100 kHz]
```

where `P_k = 10^(PSD_k / 10)` (linear power, not dB).

- Rationale: the true carrier frequency may fall between FFT bins. A
  power-weighted centroid recovers sub-bin accuracy without increasing RBW.

---

### Stage 6 — ANE database matching

- Load the ANE (Agencia Nacional del Espectro) FM broadcast station table for
  the relevant geographic area. Each record provides a licensed nominal
  frequency `f_nominal`.
- For each centroid `f_centroid`, find the nearest entry in the ANE table.
- Accept the match only if:

  ```
  |f_centroid − f_nominal| < match_tolerance    (suggested: 100 kHz)
  ```

- Compute the raw frequency error for each accepted match:

  ```
  Δf_i = f_centroid_i − f_nominal_i
  ```

- Discard unmatched candidates (no licensed station within tolerance — likely
  an unlicensed or foreign signal).

---

### Stage 7 — Regression-based PPM estimation with outlier rejection

This is the core estimation stage. Rather than averaging per-peak PPM values
independently, fit a **zero-intercept linear model** across all matched
stations in a single cycle:

```
Δf_i = α · f_nominal_i
```

**Step 7a — Initial least-squares fit**

```
α_hat = Σ(Δf_i · f_nominal_i) / Σ(f_nominal_i²)     (closed-form OLS, zero intercept)
ppm_estimate = α_hat × 1e6
```

**Step 7b — Residual computation and outlier flagging**

```
residual_i = Δf_i − α_hat · f_nominal_i
```

Flag any station where `|residual_i| > 3 kHz` as a corrupted observation.
Likely causes: multipath, adjacent-channel bleed, modulation asymmetry,
RDS/pilot subcarrier bias.

**Step 7c — Refit on clean subset**

- Remove all flagged observations.
- Recompute `α_hat` on the remaining stations.
- Require at least **3 clean stations** for the fit to be considered valid.
  If fewer than 3 remain, discard the entire cycle and log a quality warning.

**Step 7d — Goodness-of-fit check**

Compute R² for the zero-intercept model on the clean subset:

```
SS_res = Σ(residual_i²)
SS_tot = Σ(Δf_i²)
R² = 1 − SS_res / SS_tot
```

- Accept the cycle estimate only if **R² ≥ 0.90**.
- A low R² after outlier removal indicates that measurement conditions are
  poor (heavy multipath, sparse station density) and calibration should be
  deferred.

---

### Stage 8 — Multi-cycle convergence loop

- Repeat Stages 1–7 for **M = 10** independent cycles, collecting one
  `ppm_estimate` per accepted cycle.
- After each new accepted estimate, compute the rolling statistics:

  ```
  ppm_mean   = mean of accepted estimates so far
  ppm_stderr = std / sqrt(n_accepted)
  ```

- Declare **convergence** when `ppm_stderr < 0.5 PPM` and at least
  **5 cycles** have been accepted.
- If convergence is not reached after 10 cycles, extend by up to 5 additional
  cycles before halting with a warning.

---

### Stage 9 — Clock correction write-back

- Apply the converged `ppm_mean` to the HackRF One frequency correction
  parameter (via `ppm_error` in `ServerRealtimeConfig` or the equivalent
  driver call).
- Log the correction with a timestamp, the number of contributing stations,
  the number of accepted cycles, and the final `ppm_stderr`.

---

### Stage 10 — Periodic re-calibration

- Crystal oscillator error drifts with **temperature and aging**. Schedule
  re-calibration:
  - At session start (cold oscillator).
  - After any significant ambient temperature change (> 5 °C).
  - Every **60 minutes** during continuous monitoring sessions.
- Store the time series of PPM estimates to detect long-term drift trends.

---

## Quality gates summary

| Gate | Condition | Action on failure |
|---|---|---|
| Bandwidth filter | 50 kHz ≤ BW ≤ 300 kHz | Discard candidate |
| ANE match tolerance | \|Δf\| < 100 kHz | Discard candidate |
| Outlier rejection | \|residual\| < 3 kHz | Remove from fit |
| Minimum stations | n_clean ≥ 3 | Discard cycle |
| Goodness of fit | R² ≥ 0.90 | Discard cycle |
| Convergence | stderr < 0.5 PPM, n ≥ 5 | Extend or halt |

---

## Diagnostic outputs (per cycle)

- PSD plot with detected peaks, centroids, and bandwidth shading.
- Regression plot: `Δf` vs `f_nominal` with fitted line, residuals, and
  flagged outliers highlighted.
- Per-station table: `f_nominal`, `f_centroid`, `Δf`, `residual`, `status`.
- Rolling PPM estimate with confidence band across cycles.
````

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_widths
import pandas as pd
from dataclasses import dataclass, asdict
import time

# Patch notebook jupyter
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import nest_asyncio
nest_asyncio.apply()

import cfg
log = cfg.set_logger()

from functions import AcquireDual
from utils import ZmqPairController, ServerRealtimeConfig

@dataclass
class DspPayload:
    start_freq_hz: float
    end_freq_hz: float
    Pxx: np.ndarray

    @classmethod
    def from_dict(cls, data: dict):
        return cls(**{k: v for k, v in data.items() if k in cls.__dataclass_fields__})


async def get_rf_payload(config: ServerRealtimeConfig) -> DspPayload:
    """Abre la conexión, adquiere los datos y retorna el objeto listo."""
    controller = ZmqPairController(addr=cfg.IPC_ADDR, is_server=True, verbose=False)
    async with controller as zmq_ctrl:
        acquirer = AcquireDual(controller=zmq_ctrl, log=log)
        raw_data = await acquirer.get_corrected_data(asdict(config))
        return DspPayload.from_dict(raw_data)

async def get_representative_psd(config: ServerRealtimeConfig, iterations: int = 10) -> DspPayload:
    """Realiza N capturas y devuelve un DspPayload con el PSD Max Hold estabilizado."""
    psd_captures = []
    latest_payload : DspPayload
    
    for i in range(iterations):
        log.info(f"Capturing PSD [[{i + 1}/{iterations}]]")
        latest_payload = await get_rf_payload(config)
        psd_captures.append(latest_payload.Pxx)
        
    # Apilar todas las capturas y calcular el Max Hold
    psd_matrix = np.array(psd_captures)
    psd_max_hold = np.max(psd_matrix, axis=0)
    
    log.info("Capturas finalizadas. PSD representativo calculado.")
    
    # Retornamos un nuevo objeto con el PSD estabilizado y las frecuencias correctas
    return DspPayload(
        start_freq_hz=latest_payload.start_freq_hz,
        end_freq_hz=latest_payload.end_freq_hz,
        Pxx=psd_max_hold
    )

def mask_dc(psd_array: np.ndarray) -> np.ndarray:
    psd_search = psd_array.copy()
    center_idx = len(psd_search) // 2
    ignore_width = int(len(psd_search) * 0.025)
    psd_search[center_idx - ignore_width : center_idx + ignore_width] = -np.inf
    return psd_search

def floor_and_thres(psd_array: np.ndarray) -> tuple:
    noise_floor = np.percentile(psd_array, 10)
    pxx_range = np.max(psd_array) - np.min(psd_array)
    threshold = noise_floor + (0.01 * pxx_range)
    return noise_floor, threshold

def validate_peaks(psd_search: np.ndarray, psd_real: np.ndarray, freq: np.ndarray, 
                   bin_size_hz: float, threshold: float, number_candidates: int = 3) -> list:
    peaks, props = find_peaks(psd_search, height=threshold, distance=40)
    widths, _, left_ips, right_ips = peak_widths(psd_search, peaks, rel_height=0.8)
    widths_hz = widths * bin_size_hz
    
    valid_mask = (widths_hz > 50e3) & (widths_hz < 300e3)
    orig_indices = np.where(valid_mask)[0]
    valid_heights = props['peak_heights'][valid_mask]
    
    top_sort = np.argsort(valid_heights)[-number_candidates:]
    top_orig_indices = orig_indices[top_sort]
        
    results = []
    for orig_idx in top_orig_indices:
        idx = peaks[orig_idx]
        window_bins = int(100e3 / bin_size_hz)
        window = slice(max(0, idx - window_bins), min(len(freq), idx + window_bins))
        power_lin = 10**(psd_real[window] / 10)
        f_center = np.sum(freq[window] * power_lin) / np.sum(power_lin)
        
        # Extraer límites de ancho de banda en MHz
        left_mhz = freq[int(left_ips[orig_idx])]
        right_mhz = freq[int(right_ips[orig_idx])]
        
        results.append((idx, f_center, left_mhz, right_mhz))
        
    return results

# -----------------------------------------------------------

async def main():
    log.info("Starting FM calibration...")
    config_obj = ServerRealtimeConfig(
        method_psd="welch", center_freq_hz=int(98e6), sample_rate_hz=int(20e6),
        rbw_hz=int(10e3), window="hamming", overlap=float(0.5), lna_gain=int(8),
        vga_gain=int(0), antenna_amp=bool(False), antenna_port=int(1), ppm_error=0,
    )

    rf = await get_representative_psd(config_obj, iterations=5)
    freq = np.linspace(rf.start_freq_hz, rf.end_freq_hz, len(rf.Pxx)) / 1e6
    bin_size_hz = (rf.end_freq_hz - rf.start_freq_hz) / len(rf.Pxx)
    
    # Analysis steps
    psd_search = mask_dc(rf.Pxx)
    noise_floor, threshold = floor_and_thres(rf.Pxx)
    validated_data = validate_peaks(psd_search, rf.Pxx, freq, bin_size_hz, threshold, number_candidates=3)
    
    # Plotting
    plt.figure(figsize=(20, 8))
    plt.plot(freq, rf.Pxx, color='darkorange', linewidth=1.2, label='PSD Max Hold')
    plt.axhline(noise_floor, color='blue', linestyle='--', alpha=0.5, label=f'Noise Floor ({noise_floor:.1f} dBm)')
    plt.axhline(threshold, color='green', linestyle='-.', alpha=0.7, label=f'Threshold ({threshold:.1f} dBm)')
    
    if validated_data:
        indices = [item[0] for item in validated_data]
        plt.plot(freq[indices], rf.Pxx[indices], "rx", markersize=12, markeredgewidth=2, label='Peak Bins')
        
        for idx, centroid_mhz, left_mhz, right_mhz in validated_data:
            # Línea y texto del centroide
            plt.axvline(centroid_mhz, color='red', linestyle=':', alpha=0.8)
            plt.text(centroid_mhz, rf.Pxx[idx] + 2, f"{centroid_mhz:.4f} MHz", color='red', fontsize=10, ha='center')
            
            # Sombra del ancho de banda relativo
            plt.axvspan(left_mhz, right_mhz, color='yellow', alpha=0.3, label='Bandwidth' if idx == indices[0] else "")

    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Power Spectral Density (dBm)")
    plt.title("FM Calibration - Validated Peaks & Centroids")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return 0

"""
if __name__ == "__main__":
    now = time.perf_counter()  # Warm up the timer
    rc = cfg.run_and_capture(main)
    final_time = time.perf_counter() - now
    log.info(f"FM calibration completed with return code: {rc}")
    log.info(f"Total execution time: {final_time:.2f} seconds")
"""

'\nif __name__ == "__main__":\n    now = time.perf_counter()  # Warm up the timer\n    rc = cfg.run_and_capture(main)\n    final_time = time.perf_counter() - now\n    log.info(f"FM calibration completed with return code: {rc}")\n    log.info(f"Total execution time: {final_time:.2f} seconds")\n'

In [2]:
def haversine_m(lat1, lon1, lat2, lon2):
    # Radio de la Tierra en metros
    R = 6371000.0 
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

def find_coverage_FM(ane_db, coverage_area_m, lat, lng):
    df = ane_db[ane_db["servicio"].astype(str).str.strip() == "Radiodifusión Sonora en FM"].copy()
    df = df.dropna(subset=["frecuencia", "latitud_dec", "longitud_dec"])
    
    # Cálculo y filtrado en metros
    df["dist_m"] = haversine_m(lat, lng, df["latitud_dec"].to_numpy(float), df["longitud_dec"].to_numpy(float))
    
    filtered = df[df["dist_m"] <= coverage_area_m].sort_values("dist_m")
    
    log.info(filtered[["codigo_dane", "frecuencia", "dist_m"]].to_string(index=False))
    return filtered["frecuencia"].unique().tolist()

# Configuración
dummy_lat, dummy_lng = 5.0310736, -75.5894066
COVERAGE_AREA_M = 10000

# Carga y ejecución
ane_db = pd.read_csv(cfg.PROJECT_ROOT / "db" / "ANE_db_reference.csv")
frecuencias = find_coverage_FM(ane_db, COVERAGE_AREA_M, dummy_lat, dummy_lng)

log.info(f"\nFrecuencias en radio de {COVERAGE_AREA_M}m:")
log.info(frecuencias)

21-Mar-26(22:10:03)[IPYKERNEL_LAUNCHER]INFO       codigo_dane  frecuencia      dist_m
       17524       106.1 3956.721773
       17524        97.2 4105.252199
       17174       104.1 7053.996205
       17174        89.1 7054.015723
       17873       107.3 8148.528560
       17001        90.3 8495.066702
21-Mar-26(22:10:03)[IPYKERNEL_LAUNCHER]INFO      
Frecuencias en radio de 10000m:
21-Mar-26(22:10:03)[IPYKERNEL_LAUNCHER]INFO      [106.1, 97.2, 104.1, 89.1, 107.3, 90.3]


/tmp/ipykernel_29372/2465063677.py:32: DtypeWarning: Columns (0: potencia, 1: distintivo, 2: expediente, 3: latitud_dec, 4: longitud_dec, 5: fecha_inicial, 6: fecha_final) have mixed types. Specify dtype option on import or set low_memory=False.
  ane_db = pd.read_csv(cfg.PROJECT_ROOT / "db" / "ANE_db_reference.csv")
